In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('users_commodities_india_geo.csv')
df['commodity_list'] = df['commodity'].apply(
    lambda x: [c.strip() for c in x.split(';')]
)
print(df.shape)
df.head()

(1000, 12)


,user_id,commodity,role,city,state,latitude_raw,longitude_raw,latitude_normalized,longitude_normalized,min_quantity_mt,max_quantity_mt,commodity_list
0,1,sugar;rice;cotton,trader,Dindigul,Tamil Nadu,10.3673,77.9757,-0.824644,-0.312021,63,285,"[sugar, rice, cotton]"
1,2,rice,trader,Mumbai,Maharashtra,19.0760,72.8777,-0.179556,-0.663607,61,354,[rice]
2,3,cotton;rice,exporter,Indore,Madhya Pradesh,22.7196,75.8577,0.090341,-0.458090,55,203,"[cotton, rice]"
3,4,rice;sugar,trader,Gandhinagar,Gujarat,23.2156,72.6369,0.127081,-0.680214,99,380,"[rice, sugar]"
4,5,cotton;sugar,exporter,Ludhiana,Punjab,30.9010,75.8573,0.696370,-0.458117,73,188,"[cotton, sugar]"


In [2]:
ALL_COMMODITIES = sorted(set(
    c for lst in df['commodity_list'] for c in lst
))
print("Commodities found:", ALL_COMMODITIES)

COMMODITY_BOOST = 0.9

def encode_commodity(commodity_list):
    vec = np.zeros(len(ALL_COMMODITIES))
    for c in commodity_list:
        if c in ALL_COMMODITIES:
            vec[ALL_COMMODITIES.index(c)] = 1.0
    return vec * COMMODITY_BOOST

commodity_vecs = np.array([
    encode_commodity(row['commodity_list'])
    for _, row in df.iterrows()
])

print("Dimension names:", [f"has_{c}" for c in ALL_COMMODITIES])
print()
for i in range(5):
    row = df.iloc[i]
    raw = commodity_vecs[i] / COMMODITY_BOOST
    vals = dict(zip([f"has_{c}" for c in ALL_COMMODITIES], raw))
    print(f"User {row['user_id']} | {row['commodity']:20s} → {vals}")

Commodities found: ['cotton', 'rice', 'sugar']
Dimension names: ['has_cotton', 'has_rice', 'has_sugar']

User 1 | sugar;rice;cotton    → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 2 | rice                 → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(0.0)}
User 3 | cotton;rice          → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(0.0)}
User 4 | rice;sugar           → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 5 | cotton;sugar         → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(0.0), 'has_sugar': np.float64(1.0)}


In [3]:
# ─── Quantity Encoding ───────────────────────────────────────────────────
# Each user has a [min_quantity_mt, max_quantity_mt] range.
# We normalise both values to [0, 1] with a global MinMaxScaler so that
# users with similar quantity ranges produce similar vectors.
#
# CANDIDATE vector  →  encode_quantity(qty_min, qty_max)   (stored in DB)
# QUERY vector      →  same function, searcher's own range  (used at search time)
#
# Raise QTY_BOOST to make quantity overlap matter more in the final score.
# ─────────────────────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler

QTY_BOOST = 3.0  # ← tune this; higher = quantity overlap counts more

# Fit scaler on the whole dataset so every call uses the same scale
_qty_raw = df[['min_quantity_mt', 'max_quantity_mt']].values
qty_scaler = MinMaxScaler()
qty_scaler.fit(_qty_raw)            # learn global min/max — do NOT transform here

QTY_DIMS = ['qty_min_norm', 'qty_max_norm']

def encode_quantity(qty_min, qty_max, boost=QTY_BOOST):
    """
    Normalise a [min, max] quantity range to [0,1] using the globally
    fitted qty_scaler, then apply the boost weight.

    Used identically for both stored candidate vectors and query vectors,
    so cosine similarity rewards users whose quantity ranges overlap.
    """
    scaled = qty_scaler.transform([[qty_min, qty_max]])[0]
    return scaled * boost

qty_vecs = np.array([
    encode_quantity(row['min_quantity_mt'], row['max_quantity_mt'])
    for _, row in df.iterrows()
])

print(f"Quantity vector shape : {qty_vecs.shape}")
print(f"QTY_BOOST             : {QTY_BOOST}")
print(f"Dimension names       : {QTY_DIMS}")
print()
print("Sample encodings (normalised, before boost):")
for i in range(5):
    row  = df.iloc[i]
    raw  = qty_vecs[i] / QTY_BOOST
    print(f"  User {row['user_id']:3d} | "
          f"{row['min_quantity_mt']:3.0f}–{row['max_quantity_mt']:3.0f} mt  →  "
          f"qty_min_norm={raw[0]:.4f}  qty_max_norm={raw[1]:.4f}")


Quantity vector shape : (1000, 2)
QTY_BOOST             : 3.0
Dimension names       : ['qty_min_norm', 'qty_max_norm']

Sample encodings (normalised, before boost):
  User   1 |  63–285 mt  →  qty_min_norm=0.5375  qty_max_norm=0.6460
  User   2 |  61–354 mt  →  qty_min_norm=0.5125  qty_max_norm=0.8602
  User   3 |  55–203 mt  →  qty_min_norm=0.4375  qty_max_norm=0.3913
  User   4 |  99–380 mt  →  qty_min_norm=0.9875  qty_max_norm=0.9410
  User   5 |  73–188 mt  →  qty_min_norm=0.6625  qty_max_norm=0.3447


In [4]:
ROLE_AFFINITY = {
    'trader':   {'want_broker': 0.55, 'want_exporter': 0.3, 'want_trader': 0.2},
    'broker':   {'want_broker': 0.33,'want_exporter': 0.33,'want_trader': 0.33},
    'exporter': {'want_broker': 0.7, 'want_exporter': 0.2, 'want_trader': 0.3},
}

# What each role OFFERS — used for candidate vectors
ROLE_OFFERS = {
    'trader':   {'is_broker': 0.0, 'is_exporter': 0.0, 'is_trader': 1.0},
    'broker':   {'is_broker': 1.0, 'is_exporter': 0.0, 'is_trader': 0.0},
    'exporter': {'is_broker': 0.0, 'is_exporter': 1.0, 'is_trader': 0.0},
}

ROLE_DIMS  = ['want_broker', 'want_exporter', 'want_trader']
ROLE_BOOST = 1.5

def encode_role_searcher(role):
    """What this user WANTS — used for the query vector."""
    affinity = ROLE_AFFINITY[role]
    return np.array([affinity[d] for d in ROLE_DIMS]) * ROLE_BOOST

def encode_role_candidate(role):
    offers = ROLE_OFFERS[role]
    return np.array([
        offers['is_broker'],
        offers['is_exporter'],
        offers['is_trader']
    ]) * ROLE_BOOST

# Stored vectors use candidate encoding (what they ARE)
role_vecs = np.array([
    encode_role_candidate(row['role'])
    for _, row in df.iterrows()
])

print("Dimension names:", ROLE_DIMS)
print()
print("Candidate vectors (what they ARE — stored in DB):")
for role in ['trader', 'broker', 'exporter']:
    v = encode_role_candidate(role) / ROLE_BOOST
    print(f"  {role:10s} → {dict(zip(ROLE_DIMS, v))}")

print()
print("Searcher vectors (what they WANT — used at query time only):")
for role in ['trader', 'broker', 'exporter']:
    v = encode_role_searcher(role) / ROLE_BOOST
    print(f"  {role:10s} → {dict(zip(ROLE_DIMS, v))}")

print()
print("Dot product preview for trader searching:")
trader_want = encode_role_searcher('trader')
for role in ['broker', 'exporter', 'trader']:
    candidate_offer = encode_role_candidate(role)
    dot = np.dot(trader_want, candidate_offer)
    print(f"  trader → {role}: dot = {dot:.3f}")

Dimension names: ['want_broker', 'want_exporter', 'want_trader']

Candidate vectors (what they ARE — stored in DB):
  trader     → {'want_broker': np.float64(0.0), 'want_exporter': np.float64(0.0), 'want_trader': np.float64(1.0)}
  broker     → {'want_broker': np.float64(1.0), 'want_exporter': np.float64(0.0), 'want_trader': np.float64(0.0)}
  exporter   → {'want_broker': np.float64(0.0), 'want_exporter': np.float64(1.0), 'want_trader': np.float64(0.0)}

Searcher vectors (what they WANT — used at query time only):
  trader     → {'want_broker': np.float64(0.55), 'want_exporter': np.float64(0.3), 'want_trader': np.float64(0.20000000000000004)}
  broker     → {'want_broker': np.float64(0.33), 'want_exporter': np.float64(0.33), 'want_trader': np.float64(0.33)}
  exporter   → {'want_broker': np.float64(0.6999999999999998), 'want_exporter': np.float64(0.20000000000000004), 'want_trader': np.float64(0.3)}

Dot product preview for trader searching:
  trader → broker: dot = 1.238
  trader → ex

In [5]:
# city_enc  = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# state_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# city_vecs  = city_enc.fit_transform(df[['city']])
# state_vecs = state_enc.fit_transform(df[['state']])

# print("City dimensions:",  city_enc.categories_[0].tolist())
# print("State dimensions:", state_enc.categories_[0].tolist())
# print()
# print("City vec shape:", city_vecs.shape)
# print("State vec shape:", state_vecs.shape)

import numpy as np

def encode_geo_3d(lat, lon, boost=1.0):
    """
    Normalizes raw Lat/Lon into 3D Cartesian coordinates.
    Works perfectly with Cosine Similarity.
    """
    # 1. Convert degrees to radians
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    
    # 2. Project to 3D Space (X, Y, Z)
    # This places the point on a unit sphere (radius 1)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    
    # 3. Apply weight/boost
    return np.array([x, y, z]) * boost

GEO_BOOST = 3.0 # Increase this if you want distance to be more important

geo_vec_list = []
for idx, row in df.iterrows():
    vec = encode_geo_3d(row['latitude_raw'], row['longitude_raw'], GEO_BOOST)
    geo_vec_list.append(vec)

geo_vecs = np.array(geo_vec_list)

In [6]:
# Final vector layout:
# [ has_cotton | has_rice | has_sugar | role_dims(3) | geo(3) | qty_min_norm | qty_max_norm ]

master_vectors = np.hstack([
    commodity_vecs,   # has_X dims      (n_comm)
    role_vecs,        # role dims        (n_role)
    geo_vecs,         # 3D cartesian geo (3)
    qty_vecs,         # qty_min_norm, qty_max_norm  (2)  ← NEW
])

df['vector'] = master_vectors.tolist()

# Dimension layout
n_comm  = commodity_vecs.shape[1]
n_role  = role_vecs.shape[1]
n_geo   = geo_vecs.shape[1]
n_qty   = qty_vecs.shape[1]          # ← NEW
total   = master_vectors.shape[1]

print(f"Total vector dims: {total}")
print(f"  [0 : {n_comm}]                    → commodity  ({[f'has_{c}' for c in ALL_COMMODITIES]})")
print(f"  [{n_comm} : {n_comm+n_role}]              → role       ({ROLE_DIMS})")
print(f"  [{n_comm+n_role} : {n_comm+n_role+n_geo}]          → geo        (3 dims – cartesian xyz)")
print(f"  [{n_comm+n_role+n_geo} : {total}]          → quantity   ({QTY_DIMS})  ← NEW")


Total vector dims: 11
  [0 : 3]                    → commodity  (['has_cotton', 'has_rice', 'has_sugar'])
  [3 : 6]              → role       (['want_broker', 'want_exporter', 'want_trader'])
  [6 : 9]          → geo        (3 dims – cartesian xyz)
  [9 : 11]          → quantity   (['qty_min_norm', 'qty_max_norm'])  ← NEW


In [7]:
# Build the in-memory vector DB.
# Each record stores the full master vector + raw metadata.
# qty_min / qty_max are BOTH in the vector (via qty_vecs) AND in metadata
# so they can be used for hard-filter pre-filtering at query time if needed.
vector_db = []

for idx, row in df.iterrows():
    vector_db.append({
        "id":     int(row['user_id']),
        "vector": master_vectors[idx],
        "meta": {
            "user_id":        int(row['user_id']),
            "role":           row['role'],
            "commodity":      row['commodity'],
            "commodity_list": row['commodity_list'],
            "city":           row['city'],
            "state":          row['state'],
            "latitude_raw":   row['latitude_raw'],
            "longitude_raw":  row['longitude_raw'],
            "qty_min":        int(row['min_quantity_mt']),
            "qty_max":        int(row['max_quantity_mt']),
        }
    })

print(f"Vector DB: {len(vector_db)} records")
print("\nSample:")
r = vector_db[0]
print(f"  id: {r['id']}")
print(f"  meta: {r['meta']}")
print(f"  vector dims: {len(r['vector'])}")
print(f"  vector (commodity+role section): {r['vector'][:n_comm+n_role]}")
print(f"  vector (qty section)           : {r['vector'][-n_qty:]}")
print(f"  names: {[f'has_{c}' for c in ALL_COMMODITIES] + ROLE_DIMS + QTY_DIMS}")


Vector DB: 1000 records

Sample:
  id: 1
  meta: {'user_id': 1, 'role': 'trader', 'commodity': 'sugar;rice;cotton', 'commodity_list': ['sugar', 'rice', 'cotton'], 'city': 'Dindigul', 'state': 'Tamil Nadu', 'latitude_raw': 10.3673, 'longitude_raw': 77.9757, 'qty_min': 63, 'qty_max': 285}
  vector dims: 11
  vector (commodity+role section): [0.9 0.9 0.9 0.  0.  1.5]
  vector (qty section)           : [1.6125    1.9378882]
  names: ['has_cotton', 'has_rice', 'has_sugar', 'want_broker', 'want_exporter', 'want_trader', 'qty_min_norm', 'qty_max_norm']


In [22]:
!pip install plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.9 MB 5.5 MB/s eta 0:00:02
   ----- ---------------------------------- 1.3/9.9 MB 3.2 MB/s eta 0:00:03
   --------- ------------------------------ 2.4/9.9 MB 3.2 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/9.9 MB 3.5 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 3.6 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.9 MB 3.5 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.9 MB 3.5 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.9 MB 3.5 MB/s eta 0:00:02
   -------------------------- ------------- 6.6/9.9 MB 3.3 MB/s eta 0:00:02
   ---------------------------- ----------- 7.1/9.9 MB 3.2 MB/s eta 0:00:01
   ------------------------------- -------- 7.9/9.9 MB 3.3 MB/s eta 0:00:01
   -----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

DIM_LABELS = (
    [f'has_{c}' for c in ALL_COMMODITIES] +
    ROLE_DIMS +
    ['geo_x', 'geo_y', 'geo_z'] +
    QTY_DIMS
)

SEC_COLORS = {
    'commodity': '#7F77DD',
    'role':      '#1D9E75',
    'geo':       '#EF9F27',
    'qty':       '#D4537E',
}

SEC_SLICES = {
    'commodity': slice(0,           n_comm),
    'role':      slice(n_comm,      n_comm + n_role),
    'geo':       slice(n_comm+n_role, n_comm + n_role + n_geo),
    'qty':       slice(n_comm+n_role+n_geo, None),
}

In [9]:
def search(searcher_id, top_k=50):
    # 1. Locate the searcher
    searcher      = next(r for r in vector_db if r['meta']['user_id'] == searcher_id)
    searcher_meta = searcher['meta']

    # 2. Build query vector components
    s_commodity = encode_commodity(searcher_meta['commodity_list'])
    s_role      = encode_role_searcher(searcher_meta['role'])
    s_geo       = encode_geo_3d(
        searcher_meta['latitude_raw'],
        searcher_meta['longitude_raw'],
        boost=GEO_BOOST
    )
    # ── NEW: include quantity in the query vector ──────────────────────────
    s_qty = encode_quantity(
        searcher_meta['qty_min'],
        searcher_meta['qty_max'],
        boost=QTY_BOOST
    )
    # ──────────────────────────────────────────────────────────────────────

    # Order must match master_vectors: commodity → role → geo → qty
    s_vec = np.concatenate([s_commodity, s_role, s_geo, s_qty]).reshape(1, -1)

    # 3. All users except searcher
    candidates = [
        r for r in vector_db
        if r['meta']['user_id'] != searcher_id
    ]

    if not candidates:
        print("No candidates found.")
        return pd.DataFrame()

    # 4. Cosine similarity
    cand_vecs = np.array([r['vector'] for r in candidates])
    sims      = cosine_similarity(s_vec, cand_vecs)[0]

    # 5. Sort and build results
    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        m = candidates[i]['meta']
        results.append({
            'user_id':    m['user_id'],
            'role':       m['role'],
            'commodity':  m['commodity'],
            'city':       m['city'],
            'state':      m['state'],
            'lat':        m.get('latitude_raw'),
            'lon':        m.get('longitude_raw'),
            'qty_range':  f"{m['qty_min']}\u2013{m['qty_max']}mt",
            'similarity': round(float(sims[i]), 4),
        })

    return pd.DataFrame(results)


In [10]:
def visualize_search(searcher_id, top_k=8):
    # ── 1. Pull data ──────────────────────────────────────────────────────────
    s_meta  = next(r['meta'] for r in vector_db if r['meta']['user_id'] == searcher_id)
    results = search(searcher_id, top_k=top_k)
    top_ids = results['user_id'].tolist()
    sims    = results['similarity'].tolist()

    # Build the QUERY vector (want-encoding, same as search())
    query_vec = np.concatenate([
        encode_commodity(s_meta['commodity_list']),
        encode_role_searcher(s_meta['role']),
        encode_geo_3d(s_meta['latitude_raw'], s_meta['longitude_raw'], boost=GEO_BOOST),
        encode_quantity(s_meta['qty_min'], s_meta['qty_max'], boost=QTY_BOOST),
    ])

    # Candidate stored vectors, ordered by rank
    cand_records = sorted(
        [r for r in vector_db if r['meta']['user_id'] in top_ids],
        key=lambda r: top_ids.index(r['meta']['user_id'])
    )
    cand_vecs   = np.array([r['vector'] for r in cand_records])
    cand_short  = [f"U{r['meta']['user_id']} {r['meta']['role']}" for r in cand_records]
    cand_hover  = [
        f"User {r['meta']['user_id']}<br>{r['meta']['role']}<br>"
        f"{r['meta']['commodity']}<br>{r['meta']['city']}, {r['meta']['state']}<br>"
        f"{r['meta']['qty_min']}–{r['meta']['qty_max']} mt"
        for r in cand_records
    ]

    # ── 2. Figure layout ──────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            '① Radar — query vs top candidates',
            '② Heatmap — every dimension',
            '③ PCA scatter — all 1000 users',
            '④ Score breakdown by section',
        ],
        specs=[[{'type': 'polar'}, {'type': 'xy'}],
               [{'type': 'xy'},    {'type': 'xy'}]],
        vertical_spacing=0.14,
        horizontal_spacing=0.10,
    )

    # ── ① RADAR ───────────────────────────────────────────────────────────────
    # Normalise each dim to [0,1] so all axes are comparable
    all_vecs = np.vstack([query_vec, cand_vecs])
    vmax     = all_vecs.max(axis=0)
    vmax[vmax == 0] = 1
    q_norm  = query_vec / vmax
    c_norms = cand_vecs / vmax

    theta = DIM_LABELS + [DIM_LABELS[0]]   # close the polygon

    fig.add_trace(go.Scatterpolar(
        r=np.append(q_norm, q_norm[0]), theta=theta,
        name=f'Query (User {searcher_id})',
        line=dict(color='#534AB7', width=2.5),
        fill='toself', fillcolor='rgba(83,74,183,0.12)',
    ), row=1, col=1)

    for i, (cv, lbl, sim) in enumerate(zip(c_norms, cand_short, sims)):
        is_top = (i == 0)
        fig.add_trace(go.Scatterpolar(
            r=np.append(cv, cv[0]), theta=theta,
            name=f'{lbl}  sim={sim:.3f}',
            line=dict(color='rgba(15,110,86,0.7)' if is_top else 'rgba(136,135,128,0.35)',
                      width=1.8 if is_top else 1),
            fill='toself' if is_top else None,
            fillcolor='rgba(15,110,86,0.08)' if is_top else None,
        ), row=1, col=1)

    # ── ② HEATMAP ─────────────────────────────────────────────────────────────
    heat_matrix = np.vstack([query_vec, cand_vecs])
    heat_labels = [f'★ Query {searcher_id}'] + cand_short

    fig.add_trace(go.Heatmap(
        z=heat_matrix,
        x=DIM_LABELS,
        y=heat_labels,
        colorscale='Purples',
        showscale=True,
        xgap=1, ygap=1,
        hovertemplate='%{y}<br>%{x}: %{z:.3f}<extra></extra>',
    ), row=1, col=2)

    # ── ③ PCA SCATTER ─────────────────────────────────────────────────────────
    all_stored = np.array([r['vector'] for r in vector_db])
    all_uids   = np.array([r['meta']['user_id'] for r in vector_db])

    pca    = PCA(n_components=2, random_state=42)
    proj   = pca.fit_transform(all_stored)
    q_proj = pca.transform(query_vec.reshape(1, -1))[0]

    top_mask  = np.isin(all_uids, top_ids)
    rest_mask = ~top_mask

    fig.add_trace(go.Scatter(
        x=proj[rest_mask, 0], y=proj[rest_mask, 1],
        mode='markers',
        marker=dict(color='rgba(136,135,128,0.2)', size=5),
        name='Other users', showlegend=False,
        hovertemplate='User %{text}<extra></extra>',
        text=all_uids[rest_mask].astype(str),
    ), row=2, col=1)

    top_proj = proj[top_mask]
    top_uids = all_uids[top_mask]
    fig.add_trace(go.Scatter(
        x=top_proj[:, 0], y=top_proj[:, 1],
        mode='markers+text',
        marker=dict(color='#1D9E75', size=10),
        text=top_uids.astype(str),
        textposition='top center',
        textfont=dict(size=9),
        name='Top matches',
        hovertemplate='User %{text}<extra></extra>',
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=[q_proj[0]], y=[q_proj[1]],
        mode='markers+text',
        marker=dict(color='#534AB7', size=16, symbol='star'),
        text=[f'Query {searcher_id}'],
        textposition='top center',
        name='Query',
    ), row=2, col=1)

    # Variance explained annotation
    var = pca.explained_variance_ratio_
    fig.add_annotation(
        x=0.27, y=0.03, xref='paper', yref='paper',
        text=f'PC1 {var[0]*100:.1f}%  PC2 {var[1]*100:.1f}% variance',
        showarrow=False, font=dict(size=10, color='gray'),
    )

    # ── ④ SCORE BREAKDOWN ─────────────────────────────────────────────────────
    q_norm_factor = np.linalg.norm(query_vec) + 1e-9

    for sec, sl in SEC_SLICES.items():
        contribs = [
            float(np.dot(query_vec[sl], cv[sl]) /
                  (q_norm_factor * np.linalg.norm(cv) + 1e-9))
            for cv in cand_vecs
        ]
        fig.add_trace(go.Bar(
            x=cand_short, y=contribs,
            name=sec,
            marker_color=SEC_COLORS[sec],
            hovertemplate=f'{sec}: %{{y:.4f}}<extra></extra>',
        ), row=2, col=2)

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        height=820,
        title=dict(
            text=(f'<b>User {searcher_id}</b> · {s_meta["role"]} · '
                  f'{s_meta["commodity"]} · {s_meta["city"]}, {s_meta["state"]} · '
                  f'{s_meta["qty_min"]}–{s_meta["qty_max"]} mt'),
            font=dict(size=13),
        ),
        barmode='stack',
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        legend=dict(font=dict(size=10), x=1.01, y=0.99),
        margin=dict(r=160),
    )

    fig.update_xaxes(tickangle=-40, tickfont=dict(size=9), row=1, col=2)
    fig.update_yaxes(tickfont=dict(size=9), row=1, col=2)
    fig.update_xaxes(tickangle=-35, tickfont=dict(size=9), row=2, col=2)

    fig.show(renderer="browser")

In [11]:
visualize_search(68,  top_k=8)   # your original test user
visualize_search(53,  top_k=8)   # another user

In [12]:
TEST_USER_ID = 53

s = next(r['meta'] for r in vector_db if r['meta']['user_id'] == TEST_USER_ID)
print(f"Searcher: User {s['user_id']} | {s['role']} | {s['commodity']} | {s['state']} |{s['latitude_raw']}, {s['longitude_raw']} | {s['qty_min']}–{s['qty_max']}mt")
print("=" * 70)

results = search(TEST_USER_ID, top_k=15)
print(results.to_string(index=False))

print("\nWhat to verify:")
print("1. Same commodity users rank highest")
print("2. For a trader: brokers outrank exporters, exporters outrank traders")
print("3. Same city users rank above different city (same commodity+role)")
print("4. Anyone with zero qty overlap is absent from results entirely")

Searcher: User 53 | exporter | cotton | Tamil Nadu |10.3673, 77.9757 | 43–124mt
 user_id   role    commodity        city          state     lat     lon qty_range  similarity
     322 broker       cotton   Bengaluru      Karnataka 12.9716 77.5946  34–171mt      0.9682
     219 broker       cotton       Katni Madhya Pradesh 23.8340 80.3969  40–146mt      0.9585
     170 broker       cotton     Vellore     Tamil Nadu 12.9689 79.1288  67–138mt      0.9584
     264 broker       cotton  Aurangabad    Maharashtra 19.8762 75.3433  64–130mt      0.9545
     310 broker       cotton        Pali      Rajasthan 25.1752 73.3216  51–136mt      0.9535
     444 broker       cotton     Navsari        Gujarat 20.9467 72.9520  56–181mt      0.9534
     698 broker       cotton      Howrah    West Bengal 22.5958 88.2636  52–141mt      0.9519
     834 broker       cotton  Aurangabad    Maharashtra 19.8762 75.3433  37–201mt      0.9509
     703 broker       cotton     Lucknow  Uttar Pradesh 26.8467 80.9462  4

## Chroma DB for testing 
 

In [32]:
import chromadb

# PersistentClient writes to disk — data survives kernel restarts.
# Change the path to wherever you want the DB files stored.
chroma_client = chromadb.PersistentClient(path="./chroma_commodity_db")

COLLECTION_NAME = 'commodity_users'

# Delete and recreate so re-runs are idempotent
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)

# hnsw:space='cosine' matches your existing cosine_similarity logic.
# ChromaDB stores cosine *distance* = 1 - similarity, so highest sim = lowest dist.
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

print(f'Collection "{COLLECTION_NAME}" ready.')


RuntimeError: Chroma is running in http-only client mode, and can only be run with 'chromadb.api.fastapi.FastAPI' or 'chromadb.api.async_fastapi.AsyncFastAPI' as the chroma_api_impl.             see https://docs.trychroma.com/guides#using-the-python-http-only-client for more information.

In [ ]:
# Upsert all vectors into ChromaDB.
#
# ChromaDB requires:
#   ids        -> list[str]
#   embeddings -> list[list[float]]
#   metadatas  -> list[dict]  (flat key-value; no nested lists)

BATCH_SIZE = 100

ids, embeddings, metadatas = [], [], []

for record in vector_db:
    m = record['meta']
    ids.append(str(record['id']))
    embeddings.append([float(v) for v in record['vector']])
    metadatas.append({
        'user_id':        m['user_id'],
        'role':           m['role'],
        'commodity':      m['commodity'],
        'commodity_list': ';'.join(m['commodity_list']),
        'city':           m['city'],
        'state':          m['state'],
        'latitude_raw':   m['latitude_raw'],    # needed for geo query-vec rebuild
        'longitude_raw':  m['longitude_raw'],
        'qty_min':        m['qty_min'],
        'qty_max':        m['qty_max'],
    })

for start in range(0, len(ids), BATCH_SIZE):
    end = start + BATCH_SIZE
    collection.upsert(
        ids=ids[start:end],
        embeddings=embeddings[start:end],
        metadatas=metadatas[start:end],
    )

print(f'Upserted {collection.count()} records into ChromaDB.')
print(f'Vector dimension: {len(embeddings[0])}')  # should now be 11


In [ ]:
def search_chroma(searcher_id, top_k=15, filter_state=True, filter_role=None):
    """
    ChromaDB-backed search. Query vector uses WANT encoding (same as search()).

    Parameters
    ----------
    searcher_id  : int  -- user_id of the searcher
    top_k        : int  -- number of results to return
    filter_state : bool -- if True, restrict results to same state as searcher
    filter_role  : str  -- if given (e.g. 'broker'), restrict candidates to that role
    """
    # 1. Pull searcher metadata from ChromaDB
    result = collection.get(ids=[str(searcher_id)], include=['metadatas'])
    if not result['ids']:
        print(f'User {searcher_id} not found in ChromaDB.')
        return pd.DataFrame()

    s_meta       = result['metadatas'][0]
    s_role       = s_meta['role']
    s_commodities = [c.strip() for c in s_meta['commodity_list'].split(';')]

    # 2. Build query vector — order must match master_vectors
    s_commodity = encode_commodity(s_commodities)
    s_role_vec  = encode_role_searcher(s_role)
    s_geo       = encode_geo_3d(
        s_meta['latitude_raw'],
        s_meta['longitude_raw'],
        boost=GEO_BOOST
    )
    # ── NEW: include quantity in the query vector ──────────────────────────
    s_qty = encode_quantity(
        s_meta['qty_min'],
        s_meta['qty_max'],
        boost=QTY_BOOST
    )
    # ──────────────────────────────────────────────────────────────────────

    query_vec = np.concatenate([s_commodity, s_role_vec, s_geo, s_qty]).tolist()

    # 3. Build metadata filter — exclude self, optionally narrow by state / role
    conditions = [{'user_id': {'$ne': searcher_id}}]
    if filter_state:
        conditions.append({'state': {'$eq': s_meta['state']}})
    if filter_role:
        conditions.append({'role': {'$eq': filter_role}})

    where_clause = conditions[0] if len(conditions) == 1 else {'$and': conditions}

    # 4. Query ChromaDB
    query_result = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        where=where_clause,
        include=['metadatas', 'distances'],
    )

    # 5. Format results.
    # ChromaDB returns cosine *distance* (0=identical, 2=opposite).
    # Convert back: similarity = 1 - distance
    rows = []
    for meta, dist in zip(query_result['metadatas'][0], query_result['distances'][0]):
        rows.append({
            'user_id':    meta['user_id'],
            'role':       meta['role'],
            'commodity':  meta['commodity'],
            'city':       meta['city'],
            'state':      meta['state'],
            'qty_range':  f"{meta['qty_min']}\u2013{meta['qty_max']}mt",
            'similarity': round(1 - dist, 4),
        })

    return pd.DataFrame(rows)


In [ ]:
# Test ChromaDB search and verify rankings match the original in-memory search
TEST_USER_ID = 53

s_meta_raw = next(r['meta'] for r in vector_db if r['meta']['user_id'] == TEST_USER_ID)
print(f"Searcher: User {TEST_USER_ID} | {s_meta_raw['role']} | "
      f"{s_meta_raw['commodity']} | {s_meta_raw['state']} | "
      f"{s_meta_raw['qty_min']}–{s_meta_raw['qty_max']}mt")
print('=' * 70)

print('\n[ChromaDB Results]')
chroma_results = search_chroma(TEST_USER_ID, top_k=15, filter_state=False)
print(chroma_results.to_string(index=False))

print('\n[Original In-Memory Results]')
original_results = search(TEST_USER_ID, top_k=15)
print(original_results.to_string(index=False))

print('\n[Verification]')
match = chroma_results['user_id'].tolist() == original_results['user_id'].tolist()
print(f'Rankings match original: {match}')


Searcher: User 53 | trader | sugar | Tamil Nadu | 32–225mt

[ChromaDB Results]
 user_id     role         commodity            city      state qty_range  similarity
      71   broker      cotton;sugar         Chennai Tamil Nadu  41–204mt      0.8278
      92   broker      sugar;cotton        Dindigul Tamil Nadu  59–278mt      0.8278
     212   broker      cotton;sugar           Erode Tamil Nadu   32–96mt      0.8278
     595   broker      sugar;cotton        Dindigul Tamil Nadu  95–366mt      0.8278
     610   broker        rice;sugar     Thoothukudi Tamil Nadu  32–286mt      0.8278
     835   broker      cotton;sugar         Madurai Tamil Nadu  80–336mt      0.8278
     905   broker        sugar;rice Tiruchirappalli Tamil Nadu  21–190mt      0.8278
     337   broker rice;sugar;cotton      Coimbatore Tamil Nadu  49–157mt      0.7665
     362   broker sugar;cotton;rice         Madurai Tamil Nadu  21–171mt      0.7665
     375   broker rice;cotton;sugar         Chennai Tamil Nadu  30–120m

c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
# Example: filter by state AND role at query time
# This is one of the key advantages over the in-memory approach
print('Brokers only, same state as searcher:')
filtered = search_chroma(TEST_USER_ID, top_k=10, filter_state=True, filter_role='broker')
print(filtered.to_string(index=False))


Brokers only, same state as searcher:
 user_id   role         commodity            city      state qty_range  similarity
      71 broker      cotton;sugar         Chennai Tamil Nadu  41–204mt      0.8278
      92 broker      sugar;cotton        Dindigul Tamil Nadu  59–278mt      0.8278
     212 broker      cotton;sugar           Erode Tamil Nadu   32–96mt      0.8278
     595 broker      sugar;cotton        Dindigul Tamil Nadu  95–366mt      0.8278
     610 broker        rice;sugar     Thoothukudi Tamil Nadu  32–286mt      0.8278
     835 broker      cotton;sugar         Madurai Tamil Nadu  80–336mt      0.8278
     905 broker        sugar;rice Tiruchirappalli Tamil Nadu  21–190mt      0.8278
     362 broker sugar;cotton;rice         Madurai Tamil Nadu  21–171mt      0.7665
     375 broker rice;cotton;sugar         Chennai Tamil Nadu  30–120mt      0.7665
     672 broker rice;sugar;cotton     Thoothukudi Tamil Nadu  45–309mt      0.7665


c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
